## Tutorial for xpi-nn-train

A lightweight toolbox for NN model train on ARM SoC devices like RaspberryPi, OrangePi, LubanCat, etc.

⚪ Minimal example

- use standard model from torchvision
- use standard dataset from torchvision
- use default trainer_args

In [ ]:
from xpi_nn_train import get_model, run_train

# Fresh train resnet18 with CIFAR10 dataset
model = get_model('resnet18')
run_train(model, 'CIFAR10')

⚪ Customize DATA_ROOT and trainer_args

In [ ]:
import os
from xpi_nn_train import get_pretrained_model, get_trainer_args, run_train

# DATA_ROOT defines where datasets are auto-downloaded
os.environ['DATA_ROOT'] = './DATA_ROOT'

# Customize training recipe
args = get_trainer_args()
args.epochs = 1
args.batch_size = 16
args.lr = 1e-3

# Finetune resnet18 with CIFAR10 dataset
model = get_pretrained_model('resnet18')
run_train(model, 'CIFAR10')

⚪ Customize model

Changing models is easy :)

In [ ]:
import torch.nn as nn

from xpi_nn_train import run_train
from xpi_nn_train.models.mlp import MLP
from xpi_nn_train.models.lenet import LeNet5
from xpi_nn_train.models.torchvision import get_model, get_pretrained_model, get_mbv3_small

# use library provided models
model = MLP(in_dim=28*28, out_dim=10, hid_dims=[256, 32], act=nn.SiLU)
model = LeNet5(num_classes=10)
model = get_model('resnet18')
model = get_pretrained_model('resnet50')
model = get_mbv3_small(num_classes=10)
run_train(model, 'MNIST')

# use self-defined models
class MyMLP(nn.Module):
  def __init__(self):
    super().__init__()
    self.fc1 = nn.Linear(28*28, 100)
    self.act = nn.ReLU()
    self.fc2 = nn.Linear(10, 10)
  def forward(self, x):
    x = x.flatten(1)
    x = self.fc1(x)
    x = self.act(x)
    x = self.fc2(x)
    return x
my_mlp = MyMLP()
run_train(my_mlp, 'MNIST')

my_cnn = nn.Sequential(
  nn.Conv2d(1, 4, kernel_size=5, stride=2, padding=2),  # [B, C=4, H=14, W=14]
  nn.BatchNorm2d(4),
  nn.ReLU(),
  nn.Conv2d(4, 12, kernel_size=5, stride=2, padding=2), # [B, C=12, H=7, W=7]
  nn.BatchNorm2d(12),
  nn.ReLU(),
  nn.Flatten(1),
  nn.Linear(12*7*7, 128),
  nn.ReLU(),
  nn.Linear(128, 10),
)
run_train(my_cnn, 'MNIST')

⚪ Customize dataset

Datasets are more complicated, you have to use lower-API `run_train_native` rather than `run_train`

In [ ]:
from xpi_nn_train.datasets.torchvision import get_dataloader
from xpi_nn_train.models.lenet import LeNet5
from xpi_nn_train.trainers import get_trainer_args
from xpi_nn_train.trainers.torch import run_train_native

args = get_trainer_args()

# most torchvision datasets use split="train"|"val"|"test" convention, hence literally, the "train" split is for training
train_loader = get_dataloader('MNIST', split='train', batch_size=args.batch_size, num_workers=args.num_workers)
test_loader = get_dataloader('MNIST', split='val', batch_size=args.batch_size, num_workers=args.num_workers)
# however, wierdos like EMNIST has sub-category, should specify `is_train`
train_loader = get_dataloader('EMNIST', split='letters', is_train=True, batch_size=args.batch_size, num_workers=args.num_workers)
test_loader = get_dataloader('EMNIST', split='letters', is_train=False, batch_size=args.batch_size, num_workers=args.num_workers)

# run with torch trainer
model = LeNet5(num_classes=62)   # EMNIST has 62 classes in total ("By_Class")
run_train_native(model, train_loader, test_loader, args)

⚪ Trainer backends

We recommend the native `torch` trainer backend, but `lightning` is also available (requires full installation :)

In [ ]:
# use higher API
from xpi_nn_train.trainers import get_trainer_args, run_train
args = get_trainer_args()
args.backend = 'lightning'
model = MLP()
run_train(model, 'MNIST')

# or use lower API
from xpi_nn_train.datasets.lightning import LitData
from xpi_nn_train.trainers.lightning import LitModelCls, run_train_lit
args = get_trainer_args()
litmodel = LitModelCls(model, args)
litdata = LitData('MNIST', args.batch_size, args.num_workers)
run_train_lit(litmodel, litdata, args)